In [1]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig

print("Libraries imported successfully!")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

Libraries imported successfully!
Device: CPU


In [2]:
questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What is Natural Language Processing?",
    "What is overfitting?"
]

print("Number of questions:", len(questions))

Number of questions: 5


In [3]:
chosen = [
    "Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.",
    
    "Machine Learning is a branch of Artificial Intelligence where computers learn patterns from data and use them to make predictions or decisions.",
    
    "Deep Learning is a type of Machine Learning that uses neural networks with multiple layers to learn complex patterns from data.",
    
    "Natural Language Processing is a field of Artificial Intelligence that enables computers to understand, process, and generate human language.",
    
    "Overfitting occurs when a Machine Learning model learns the training data too closely and performs poorly on new or unseen data."
]

print("Preferred responses created!")

Preferred responses created!


In [4]:
rejected = [
    "AI is when computers are intelligent. It is used for many things.",
    
    "Machine Learning is when machines learn something from computers and data. It is related to AI.",
    
    "Deep Learning is advanced learning. It uses complicated networks and is difficult.",
    
    "NLP is about computers and language. It is a complex AI topic.",
    
    "Overfitting means the model fits the data too much. It can happen in Machine Learning."
]

print("Less-preferred responses created!")

Less-preferred responses created!


In [5]:
dpo_data = [
    {
        "prompt": q,
        "chosen": c,
        "rejected": r
    }
    for q, c, r in zip(questions, chosen, rejected)
]

dpo_dataset = Dataset.from_list(dpo_data)

print(dpo_dataset)
print("\nFirst example:")
print(dpo_dataset[0])

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 5
})

First example:
{'prompt': 'What is Artificial Intelligence?', 'chosen': 'Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.', 'rejected': 'AI is when computers are intelligent. It is used for many things.'}


In [6]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

print("TinyLlama loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TinyLlama loaded successfully!


In [7]:
question = "Explain the difference between Artificial Intelligence and Machine Learning."

prompt = f"### Question:\n{question}\n\n### Answer:\n"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

before_dpo = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Question:")
print(question)

print("\nBefore DPO:")
print(before_dpo)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Explain the difference between Artificial Intelligence and Machine Learning.

Before DPO:
### Question:
Explain the difference between Artificial Intelligence and Machine Learning.

### Answer:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that are typically done by humans. AI is based on the principles of logic, computation, and programming. Machine Learning (ML) is a subset of AI that


In [8]:
dpo_config = DPOConfig(
    output_dir="./dpo_model",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    use_cpu=True
)

print("DPO configuration ready!")

DPO configuration ready!


In [9]:
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer
)

print("DPO Trainer created successfully!")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DPO Trainer created successfully!


In [10]:
print("DPO training started...")

trainer.train()

print("\nDPO training completed!")

DPO training started...


Step,Training Loss
1,0.697935
2,0.097889
3,0.005760
4,0.000434
5,0.047387



DPO training completed!


In [11]:
dpo_model_path = "./dpo_academic_model"

model.save_pretrained(dpo_model_path)
tokenizer.save_pretrained(dpo_model_path)

print("DPO model saved successfully!")
print("Saved at:", dpo_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DPO model saved successfully!
Saved at: ./dpo_academic_model


In [12]:
new_question = "Why is Python useful in Artificial Intelligence?"

new_prompt = f"### Question:\n{new_question}\n\n### Answer:\n"

inputs = tokenizer(new_prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

after_dpo = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("New Question:")
print(new_question)

print("\nAfter DPO:")
print(after_dpo)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


New Question:
Why is Python useful in Artificial Intelligence?

After DPO:
### Question:
Why is Python useful in Artificial Intelligence?

### Answer:
Python





























































In [13]:
print("DPO RESPONSE COMPARISON")
print("=======================")

print("\nQuestion:")
print(question)

print("\nBefore DPO:")
print(before_dpo)

print("\nAfter DPO:")
print(after_dpo)

DPO RESPONSE COMPARISON

Question:
Explain the difference between Artificial Intelligence and Machine Learning.

Before DPO:
### Question:
Explain the difference between Artificial Intelligence and Machine Learning.

### Answer:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that are typically done by humans. AI is based on the principles of logic, computation, and programming. Machine Learning (ML) is a subset of AI that

After DPO:
### Question:
Why is Python useful in Artificial Intelligence?

### Answer:
Python





























































In [15]:
import pandas as pd

comparison = {
    "Question": [question],
    "Before DPO": [before_dpo],
    "After DPO": [after_dpo]
}

df = pd.DataFrame(comparison)

df.to_csv(
    "dpo_before_after_comparison.csv",
    index=False
)

print("Comparison saved successfully!")
print("File: dpo_before_after_comparison.csv")

df

Comparison saved successfully!
File: dpo_before_after_comparison.csv


,Question,Before DPO,After DPO
0,Explain the difference between Artificial Inte...,### Question:\nExplain the difference between ...,### Question:\nWhy is Python useful in Artific...


In [16]:
print("CHANGES AFTER DPO")
print("=================")

print("""
1. The model is trained to prefer clear and accurate responses.

2. Preferred responses are more suitable for undergraduate students.

3. DPO helps the model learn the preferred response style.

4. The responses can become more structured and relevant after
   preference optimization.

5. The main purpose of DPO is to align the model's responses
   with human preferences.
""")

CHANGES AFTER DPO

1. The model is trained to prefer clear and accurate responses.

2. Preferred responses are more suitable for undergraduate students.

3. DPO helps the model learn the preferred response style.

4. The responses can become more structured and relevant after
   preference optimization.

5. The main purpose of DPO is to align the model's responses
   with human preferences.

